# Static Airfoil Model for a 22 ft Sailboat

This notebook models a sail as a static airfoil and computes aerodynamic **lift** and **drag** as functions of:

- angle of attack `alpha`
- wind speed `V`

It uses reasonable default dimensions for a representative 22-foot cruising sailboat.


## Assumptions

- Sail height: `8.2 m`
- Sail foot: `3.1 m`
- Sail area: triangular approximation `0.5 * height * foot = 12.71 m^2`
- Air density: `1.225 kg/m^3`
- Stall angle: `16 deg`
- Lift slope before stall: `2*pi` per radian
- High-angle drag trends toward a flat-plate style peak near `90 deg`

This is a steady-state aerodynamic model. It ignores sail twist, mast interference, heel, and hull effects.


In [ ]:
import math

import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
# Representative sail geometry for a 22 ft sailboat
SAIL_HEIGHT_M = 8.2
SAIL_FOOT_M = 3.1
SAIL_AREA_M2 = 0.5 * SAIL_HEIGHT_M * SAIL_FOOT_M
STALL_ANGLE_DEG = 16.0

print(f"Sail area: {SAIL_AREA_M2:.2f} m^2")


In [ ]:
class Sail:
    AIR_DENSITY = 1.225
    LIFT_SLOPE = 2 * math.pi
    CD0 = 0.08
    CD_MAX = 1.9
    POST_STALL_DECAY = 1.35
    PLATEAU_LIFT_FRACTION = 0.15

    def __init__(self, area: float, stall_angle_rad: float):
        self.area = area
        self.stall_angle_rad = stall_angle_rad

    def force(self, angle_rad: float, speed: float) -> tuple[float, float]:
        """Calculates resultant force for attack angle in radians and wind speed in m/s as a vector [x, y] in Newtons.
        x is aligned with the sail chord line (angle=0).
        """
        drag_force = self._drag_force(angle_rad, speed)
        lift_force = self._lift_force(angle_rad, speed)
        airflow_x, airflow_y = self._airflow_direction(angle_rad)
        lift_x, lift_y = -airflow_y, airflow_x
        force_x = drag_force * airflow_x + lift_force * lift_x
        force_y = drag_force * airflow_y + lift_force * lift_y
        return force_x, force_y

    def _dynamic_pressure(self, speed: float) -> float:
        return 0.5 * self.AIR_DENSITY * speed**2

    def _wrapped_angle(self, angle_rad: float) -> float:
        return (angle_rad + math.pi) % (2 * math.pi) - math.pi

    def _effective_attack(self, angle_rad: float) -> float:
        return abs(self._wrapped_angle(angle_rad))

    def _mirrored_attack(self, angle_rad: float) -> float:
        effective = self._effective_attack(angle_rad)
        return min(effective, math.pi - effective)

    def _lift_coefficient(self, angle_rad: float) -> float:
        wrapped = self._wrapped_angle(angle_rad)
        effective = self._effective_attack(angle_rad)
        mirrored = self._mirrored_attack(angle_rad)
        cl_stall = self.LIFT_SLOPE * self.stall_angle_rad

        cl_small = self.LIFT_SLOPE * mirrored
        cl_large = cl_stall * max(
            self.PLATEAU_LIFT_FRACTION,
            1.0 - self.POST_STALL_DECAY * (mirrored - self.stall_angle_rad),
        )
        cl_magnitude = cl_small if mirrored <= self.stall_angle_rad else cl_large

        flow_phase = math.sin(2.0 * effective)
        if abs(flow_phase) < 1e-12:
            flow_sign = 0.0
        else:
            flow_sign = 1.0 if flow_phase > 0 else -1.0

        if wrapped == 0:
            side_sign = 0.0
        else:
            side_sign = 1.0 if wrapped > 0 else -1.0

        return side_sign * flow_sign * cl_magnitude

    def _drag_coefficient(self, angle_rad: float) -> float:
        effective = self._effective_attack(angle_rad)
        return self.CD0 + (self.CD_MAX - self.CD0) * abs(math.sin(effective))

    def _lift_force(self, angle_rad: float, speed: float) -> float:
        return self._dynamic_pressure(speed) * self.area * self._lift_coefficient(angle_rad)

    def _drag_force(self, angle_rad: float, speed: float) -> float:
        return self._dynamic_pressure(speed) * self.area * self._drag_coefficient(angle_rad)

    def _airflow_direction(self, angle_rad: float) -> tuple[float, float]:
        wrapped = self._wrapped_angle(angle_rad)
        return math.cos(wrapped), math.sin(wrapped)


class Boom:
    MAX_ANGLE_RAD = math.pi / 2
    CENTER_OF_EFFORT_FRACTION = 0.5
    BOOM_MASS_KG = 18.0
    SHEET_STIFFNESS = 900.0
    ANGULAR_DAMPING = 55.0
    MAST_CROSSING_RESISTANCE = 180.0
    MAST_CONTACT_ANGLE_RAD = math.radians(12.0)

    def __init__(self, boom_length: float, min_sheet_len: float, max_sheet_len: float) -> None:
        if boom_length <= 0:
            raise ValueError("boom_length must be positive")
        if min_sheet_len < 0:
            raise ValueError("min_sheet_len must be non-negative")
        if max_sheet_len < min_sheet_len:
            raise ValueError("max_sheet_len must be greater than or equal to min_sheet_len")
        max_supported_sheet_len = min_sheet_len + self._sheet_travel_for_angle(boom_length, self.MAX_ANGLE_RAD)
        if max_sheet_len > max_supported_sheet_len:
            raise ValueError("max_sheet_len exceeds the sheet length needed for a 90 degree boom angle")
        self.boom_length = boom_length
        self.min_sheet_len = min_sheet_len
        self.max_sheet_len = max_sheet_len

    def torque(
        self,
        sail: Sail,
        wind_angle_rad: float,
        wind_speed: float,
        sheet_len: float,
        boom_angle_rad: float,
        boom_angular_velocity_rad_s: float,
    ) -> float:
        """Calculates net torque on the boom from sail force, mainsheet tension, mast contact, and damping."""
        wind_angle = self._wrapped_angle(wind_angle_rad)
        aerodynamic_torque = self._aerodynamic_torque(sail, wind_angle, wind_speed, boom_angle_rad)
        sheet_torque = self._sheet_torque(boom_angle_rad, sheet_len)
        mast_torque = self._mast_torque(boom_angle_rad, aerodynamic_torque)
        damping_torque = self._damping_torque(boom_angular_velocity_rad_s)
        return aerodynamic_torque + sheet_torque + mast_torque + damping_torque

    def moment_of_inertia(self) -> float:
        return self.BOOM_MASS_KG * self.boom_length**2 / 3.0

    def _wrapped_angle(self, angle_rad: float) -> float:
        return (angle_rad + math.pi) % (2 * math.pi) - math.pi

    def _aerodynamic_torque(self, sail: Sail, wind_angle_rad: float, wind_speed: float, boom_angle_rad: float) -> float:
        attack_angle_rad = self._wrapped_angle(wind_angle_rad - boom_angle_rad)
        force_x, force_y = sail.force(attack_angle_rad, wind_speed)
        boat_force_x, boat_force_y = self._rotate_vector(force_x, force_y, boom_angle_rad)
        arm_x, arm_y = self._center_of_effort_position(boom_angle_rad)
        return arm_x * boat_force_y - arm_y * boat_force_x

    def _sheet_torque(self, boom_angle_rad: float, sheet_len: float) -> float:
        end_x, end_y = self._boom_end_position(boom_angle_rad)
        block_x = self.boom_length
        block_y = 0.0
        sheet_dx = block_x - end_x
        sheet_dy = block_y - end_y
        sheet_distance = math.hypot(sheet_dx, sheet_dy)
        if sheet_distance < 1e-12:
            return 0.0
        clamped_sheet_len = self._clamp_sheet_len(sheet_len)
        extension = sheet_distance - clamped_sheet_len
        if extension <= 0:
            return 0.0
        tension = self.SHEET_STIFFNESS * extension
        force_scale = tension / sheet_distance
        force_x = force_scale * sheet_dx
        force_y = force_scale * sheet_dy
        return end_x * force_y - end_y * force_x

    def _mast_torque(self, boom_angle_rad: float, aerodynamic_torque: float) -> float:
        if abs(boom_angle_rad) > self.MAST_CONTACT_ANGLE_RAD:
            return 0.0
        if abs(aerodynamic_torque) < 1e-9 or abs(boom_angle_rad) < 1e-9:
            return 0.0
        if aerodynamic_torque * boom_angle_rad >= 0:
            return 0.0
        return -math.copysign(self.MAST_CROSSING_RESISTANCE, aerodynamic_torque)

    def _damping_torque(self, boom_angular_velocity_rad_s: float) -> float:
        return -self.ANGULAR_DAMPING * boom_angular_velocity_rad_s

    def _center_of_effort_position(self, boom_angle_rad: float) -> tuple[float, float]:
        radius = self.CENTER_OF_EFFORT_FRACTION * self.boom_length
        return radius * math.cos(boom_angle_rad), radius * math.sin(boom_angle_rad)

    def _boom_end_position(self, boom_angle_rad: float) -> tuple[float, float]:
        return self.boom_length * math.cos(boom_angle_rad), self.boom_length * math.sin(boom_angle_rad)

    def _clamp_sheet_len(self, sheet_len: float) -> float:
        return max(self.min_sheet_len, min(sheet_len, self.max_sheet_len))

    def _sheet_travel_for_angle(self, boom_length: float, angle_rad: float) -> float:
        return 2 * boom_length * math.sin(angle_rad / 2)

    def _rotate_vector(self, x: float, y: float, angle_rad: float) -> tuple[float, float]:
        cosine = math.cos(angle_rad)
        sine = math.sin(angle_rad)
        return x * cosine - y * sine, x * sine + y * cosine


class Sailboat:
    DEFAULT_KEEL_RESISTANCE = 1.0

    def __init__(self, sail: Sail, boom: Boom) -> None:
        self.sail = sail
        self.boom = boom
        self.boat_speed = 0.0
        self.boat_heading_rad = 0.0
        self.true_wind_speed = 0.0
        self.true_wind_angle_rad = 0.0
        self.sheet_len = boom.min_sheet_len
        self.boom_angle_state_rad = 0.0
        self.boom_angular_velocity_rad_s = 0.0
        self.time_s = 0.0
        self.keel_resistance = self.DEFAULT_KEEL_RESISTANCE

    def apparent_wind_vector_world(self) -> tuple[float, float]:
        true_wind_x, true_wind_y = self._airflow_vector_from_wind(self.true_wind_speed, self.true_wind_angle_rad)
        boat_x, boat_y = self._vector_from_polar(self.boat_speed, self.boat_heading_rad)
        return true_wind_x - boat_x, true_wind_y - boat_y

    def apparent_wind_vector_boat(self) -> tuple[float, float]:
        apparent_x, apparent_y = self.apparent_wind_vector_world()
        return self._rotate_vector(apparent_x, apparent_y, -self.boat_heading_rad)

    def apparent_wind_speed(self) -> float:
        apparent_x, apparent_y = self.apparent_wind_vector_boat()
        return math.hypot(apparent_x, apparent_y)

    def apparent_wind_angle_rad(self) -> float:
        apparent_x, apparent_y = self.apparent_wind_vector_boat()
        return math.atan2(apparent_y, apparent_x)

    def boom_angle_rad(self) -> float:
        return self.boom_angle_state_rad

    def boom_angular_acceleration_rad_s2(self) -> float:
        torque = self.net_boom_torque_nm()
        return torque / self.boom.moment_of_inertia()

    def net_boom_torque_nm(self) -> float:
        return self.boom.torque(
            self.sail,
            self.apparent_wind_angle_rad(),
            self.apparent_wind_speed(),
            self.sheet_len,
            self.boom_angle_state_rad,
            self.boom_angular_velocity_rad_s,
        )

    def step(self, dt_s: float) -> None:
        acceleration = self.boom_angular_acceleration_rad_s2()
        self.boom_angular_velocity_rad_s += acceleration * dt_s
        self.boom_angle_state_rad += self.boom_angular_velocity_rad_s * dt_s
        if self.boom_angle_state_rad > self.boom.MAX_ANGLE_RAD:
            self.boom_angle_state_rad = self.boom.MAX_ANGLE_RAD
            self.boom_angular_velocity_rad_s = 0.0
        elif self.boom_angle_state_rad < -self.boom.MAX_ANGLE_RAD:
            self.boom_angle_state_rad = -self.boom.MAX_ANGLE_RAD
            self.boom_angular_velocity_rad_s = 0.0
        self.time_s += dt_s

    def attack_angle_rad(self) -> float:
        return self._wrapped_angle(self.apparent_wind_angle_rad() - self.boom_angle_rad())

    def sail_force_sail_frame(self) -> tuple[float, float]:
        return self.sail.force(self.attack_angle_rad(), self.apparent_wind_speed())

    def sail_force_boat_frame(self) -> tuple[float, float]:
        force_x, force_y = self.sail_force_sail_frame()
        return self._rotate_vector(force_x, force_y, self.boom_angle_rad())

    def sail_force_world_frame(self) -> tuple[float, float]:
        force_x, force_y = self.sail_force_boat_frame()
        return self._rotate_vector(force_x, force_y, self.boat_heading_rad)

    def projected_force_boat_frame(self) -> tuple[float, float]:
        force_x, force_y = self.sail_force_boat_frame()
        side_scale = max(0.0, 1.0 - self.keel_resistance)
        return force_x, force_y * side_scale

    def drive_force(self) -> float:
        return -self.projected_force_boat_frame()[0]

    def side_force(self) -> float:
        return self.sail_force_boat_frame()[1]

    def _wrapped_angle(self, angle_rad: float) -> float:
        return (angle_rad + math.pi) % (2 * math.pi) - math.pi

    def _rotate_vector(self, x: float, y: float, angle_rad: float) -> tuple[float, float]:
        cosine = math.cos(angle_rad)
        sine = math.sin(angle_rad)
        return x * cosine - y * sine, x * sine + y * cosine

    def _vector_from_polar(self, magnitude: float, angle_rad: float) -> tuple[float, float]:
        return magnitude * math.cos(angle_rad), magnitude * math.sin(angle_rad)

    def _airflow_vector_from_wind(self, speed: float, from_angle_rad: float) -> tuple[float, float]:
        flow_angle_rad = self._wrapped_angle(from_angle_rad + math.pi)
        return self._vector_from_polar(speed, flow_angle_rad)


def advance_sailboat(sailboat: Sailboat, duration_s: float, dt_s: float) -> None:
    steps = max(0, int(duration_s / dt_s))
    for _ in range(steps):
        sailboat.step(dt_s)


def angle_grid_deg(start_deg: float, stop_deg: float, count: int) -> list[float]:
    if count < 2:
        return [start_deg]
    step = (stop_deg - start_deg) / (count - 1)
    return [start_deg + index * step for index in range(count)]


def angles_to_radians(angles_deg: list[float]) -> list[float]:
    return [math.radians(angle_deg) for angle_deg in angles_deg]


def decompose_force(angle_rad: float, force_xy: tuple[float, float]) -> tuple[float, float]:
    airflow_x = math.cos(angle_rad)
    airflow_y = math.sin(angle_rad)
    lift_x, lift_y = -airflow_y, airflow_x
    force_x, force_y = force_xy
    drag_force = force_x * airflow_x + force_y * airflow_y
    lift_force = force_x * lift_x + force_y * lift_y
    return drag_force, lift_force


def force_vectors_for_angles(sail: Sail, angles_rad: list[float], speed: float) -> list[tuple[float, float]]:
    return [sail.force(angle_rad, speed) for angle_rad in angles_rad]


def lift_drag_for_angles(sail: Sail, angles_rad: list[float], speed: float) -> tuple[list[float], list[float]]:
    lifts = []
    drags = []
    for angle_rad in angles_rad:
        drag_force, lift_force = decompose_force(angle_rad, sail.force(angle_rad, speed))
        lifts.append(lift_force)
        drags.append(drag_force)
    return lifts, drags


def resultant_force_magnitudes(sail: Sail, angles_rad: list[float], speed: float) -> list[float]:
    magnitudes = []
    for force_x, force_y in force_vectors_for_angles(sail, angles_rad, speed):
        magnitudes.append(math.hypot(force_x, force_y))
    return magnitudes


def resultant_force_angle_deg(sail: Sail, angles_rad: list[float], speed: float) -> list[float]:
    angles_deg = []
    for angle_rad in angles_rad:
        drag_force, lift_force = decompose_force(angle_rad, sail.force(angle_rad, speed))
        angles_deg.append(math.degrees(math.atan2(lift_force, drag_force)))
    return angles_deg


def resultant_force_angle_to_airfoil_deg(sail: Sail, angles_rad: list[float], speed: float) -> list[float]:
    airflow_angles_deg = resultant_force_angle_deg(sail, angles_rad, speed)
    return [90.0 - math.degrees(angle_rad) - airflow_angle_deg for angle_rad, airflow_angle_deg in zip(angles_rad, airflow_angles_deg)]


sail = Sail(area=SAIL_AREA_M2, stall_angle_rad=math.radians(STALL_ANGLE_DEG))
boom = Boom(
    boom_length=SAIL_FOOT_M,
    min_sheet_len=0.0,
    max_sheet_len=2 * SAIL_FOOT_M * math.sin(math.pi / 4),
)
sailboat = Sailboat(sail=sail, boom=boom)


In [ ]:
angles_deg = angle_grid_deg(0, 180, 361)
angles_rad = angles_to_radians(angles_deg)
wind_speeds = [4, 8, 12]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for wind in wind_speeds:
    lifts, drags = lift_drag_for_angles(sail, angles_rad, wind)
    axes[0].plot(angles_deg, lifts, label=f"{wind} m/s")
    axes[1].plot(angles_deg, drags, label=f"{wind} m/s")

for ax in axes:
    ax.axvline(STALL_ANGLE_DEG, color="gray", linestyle="--", linewidth=1)
    ax.axvline(90, color="gray", linestyle=":", linewidth=1)
    ax.set_xlim(0, 180)
    ax.legend()

axes[0].set_title("Lift vs Angle of Attack")
axes[0].set_xlabel("Angle of attack (deg)")
axes[0].set_ylabel("Lift (N)")
axes[1].set_title("Drag vs Angle of Attack")
axes[1].set_xlabel("Angle of attack (deg)")
axes[1].set_ylabel("Drag (N)")
plt.tight_layout()


In [ ]:
combined_wind = 8.0
resultant_force_n = resultant_force_magnitudes(sail, angles_rad, combined_wind)

plt.figure(figsize=(14, 5))
plt.plot(angles_deg, resultant_force_n, label="Resultant force magnitude", linewidth=2)
plt.axvline(STALL_ANGLE_DEG, color="gray", linestyle="--", linewidth=1)
plt.axvline(90, color="gray", linestyle=":", linewidth=1)
plt.xlim(0, 180)
plt.title(f"Resultant Aerodynamic Force vs Angle of Attack at {combined_wind} m/s")
plt.xlabel("Angle of attack (deg)")
plt.ylabel("Force magnitude (N)")
plt.legend()
plt.tight_layout()


In [ ]:
resultant_angle_deg = resultant_force_angle_deg(sail, angles_rad, combined_wind)

plt.figure(figsize=(14, 5))
plt.plot(angles_deg, resultant_angle_deg, linewidth=2, color="tab:green")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(STALL_ANGLE_DEG, color="gray", linestyle="--", linewidth=1)
plt.axvline(90, color="gray", linestyle=":", linewidth=1)
plt.xlim(0, 180)
plt.ylim(-90, 90)
plt.title("Resultant Force Angle Relative to Oncoming Airflow")
plt.xlabel("Angle of attack (deg)")
plt.ylabel("Resultant angle from drag direction (deg)")
plt.tight_layout()


In [ ]:
airfoil_angle_plot_deg = angle_grid_deg(1, 90, 180)
airfoil_angle_plot_rad = angles_to_radians(airfoil_angle_plot_deg)
resultant_angle_to_airfoil_deg = resultant_force_angle_to_airfoil_deg(sail, airfoil_angle_plot_rad, combined_wind)

plt.figure(figsize=(14, 5))
plt.plot(airfoil_angle_plot_deg, resultant_angle_to_airfoil_deg, linewidth=2, color="tab:orange")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(STALL_ANGLE_DEG, color="gray", linestyle="--", linewidth=1)
plt.axvline(90, color="gray", linestyle=":", linewidth=1)
plt.xlim(0, 90)
plt.title("Resultant Force Angle Relative to the Airfoil Chord")
plt.xlabel("Angle of attack (deg)")
plt.ylabel("Angle from airfoil chord (deg)")
plt.tight_layout()


In [ ]:
diagram_alphas_deg = [0, 16, 45, 90]
diagram_angles_rad = angles_to_radians(diagram_alphas_deg)
diagram_wind = 8.0
diagram_forces = force_vectors_for_angles(sail, diagram_angles_rad, diagram_wind)
diagram_force_scale = max(
    math.hypot(force_x, force_y)
    for force_x, force_y in diagram_forces
)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, alpha_deg, angle_rad, force_xy in zip(axes, diagram_alphas_deg, diagram_angles_rad, diagram_forces):
    drag_force, lift_force = decompose_force(angle_rad, force_xy)
    airflow_x = math.cos(angle_rad)
    airflow_y = math.sin(angle_rad)
    lift_x, lift_y = -airflow_y, airflow_x

    drag_vec = (
        airflow_x * drag_force / diagram_force_scale,
        airflow_y * drag_force / diagram_force_scale,
    )
    lift_vec = (
        lift_x * lift_force / diagram_force_scale,
        lift_y * lift_force / diagram_force_scale,
    )
    resultant_vec = (
        force_xy[0] / diagram_force_scale,
        force_xy[1] / diagram_force_scale,
    )

    ax.annotate("", xy=(1.05 * airflow_x, 1.05 * airflow_y), xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=1.0, color="tab:blue"))
    ax.annotate("", xy=drag_vec, xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2.4, color="tab:red"))
    ax.annotate("", xy=lift_vec, xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2.4, color="tab:green"))
    ax.annotate("", xy=resultant_vec, xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2.8, color="black"))

    ax.text(1.08 * airflow_x + 0.03, 1.08 * airflow_y + 0.03, "Airflow", color="tab:blue", fontsize=10)
    ax.text(drag_vec[0] + 0.03, drag_vec[1] - 0.05, "Drag", color="tab:red", fontsize=10)
    ax.text(lift_vec[0] + 0.03, lift_vec[1] + 0.03, "Lift", color="tab:green", fontsize=10)
    ax.text(resultant_vec[0] + 0.03, resultant_vec[1] + 0.03, "Resultant", color="black", fontsize=10)

    ax.set_title(f"alpha = {alpha_deg} deg")
    ax.set_aspect("equal")
    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-1.25, 1.25)
    ax.grid(True, alpha=0.25)
    ax.axhline(0, color="0.85", linewidth=1)
    ax.axvline(0, color="0.85", linewidth=1)

fig.suptitle(f"Wind and Force Vector Diagrams at {diagram_wind} m/s", fontsize=15)
plt.tight_layout()


## Notes

- Drag uses an `abs(sin(alpha))` full-angle shape, so it is smallest near `0 deg` and `180 deg`, and largest near `90 deg`.
- Lift changes sign with angle of attack and also flips after `90 deg` because the flow is effectively on the opposite side of the sail.
- Lift and drag both scale with the square of wind speed through dynamic pressure.
- The extra line plot shows the resultant aerodynamic force magnitude `sqrt(Fx^2 + Fy^2)` at a fixed wind speed.
- The airflow-relative force angle is computed by decomposing the chord-frame force back into drag and lift, then applying `atan2(lift, drag)`.
- The chord-referenced angle combines the airflow-relative force angle with the angle of attack, so it shows the force direction relative to the sail itself.


In [ ]:
exercise_boat = Sailboat(
    sail=sail,
    boom=Boom(
        boom_length=SAIL_FOOT_M,
        min_sheet_len=0.0,
        max_sheet_len=2 * SAIL_FOOT_M * math.sin(math.pi / 4),
    ),
)
exercise_boat.true_wind_speed = 8.0
exercise_boat.boat_speed = 4.0
exercise_boat.boat_heading_rad = 0.0
exercise_boat.keel_resistance = 1.0


def sheet_len_for_boom_angle_deg(boom: Boom, boom_angle_deg: float) -> float:
    boom_angle_rad = math.radians(boom_angle_deg)
    return boom.min_sheet_len + 2 * boom.boom_length * math.sin(boom_angle_rad / 2)


points_of_sail = [
    ("Irons", 0.0, 0.0),
    ("Close reach", 45.0, 20.0),
    ("Beam reach", 90.0, 45.0),
    ("Broad reach", 135.0, 70.0),
    ("Run", 180.0, 90.0),
]

print("Driving force by point of sail after boom motion settles")
print("name         wind_deg  boom_deg  attack_deg  drive_N  side_N")

for name, wind_angle_deg, target_boom_deg in points_of_sail:
    exercise_boat.boom_angle_state_rad = 0.0
    exercise_boat.boom_angular_velocity_rad_s = 0.0
    exercise_boat.true_wind_angle_rad = math.radians(wind_angle_deg)
    exercise_boat.sheet_len = min(
        exercise_boat.boom.max_sheet_len,
        sheet_len_for_boom_angle_deg(exercise_boat.boom, target_boom_deg),
    )
    advance_sailboat(exercise_boat, duration_s=6.0, dt_s=0.02)
    boom_angle_deg = math.degrees(exercise_boat.boom_angle_rad())
    attack_angle_deg = math.degrees(exercise_boat.attack_angle_rad())
    drive_force = exercise_boat.drive_force()
    side_force = exercise_boat.side_force()
    print(f"{name:12} {wind_angle_deg:8.1f} {boom_angle_deg:9.1f} {attack_angle_deg:11.1f} {drive_force:8.1f} {side_force:7.1f}")

print()
print("Gybe sweep")
print("wind_deg  boom_deg  boom_rate_deg_s")

exercise_boat.boom_angle_state_rad = 0.0
exercise_boat.boom_angular_velocity_rad_s = 0.0
exercise_boat.sheet_len = exercise_boat.boom.max_sheet_len
exercise_boat.true_wind_angle_rad = math.radians(160.0)
advance_sailboat(exercise_boat, duration_s=4.0, dt_s=0.02)

for wind_angle_deg in [160, 170, 175, 179, 180, 181, 185, 190, 200]:
    exercise_boat.true_wind_angle_rad = math.radians(wind_angle_deg)
    advance_sailboat(exercise_boat, duration_s=0.4, dt_s=0.02)
    boom_angle_deg = math.degrees(exercise_boat.boom_angle_rad())
    boom_rate_deg_s = math.degrees(exercise_boat.boom_angular_velocity_rad_s)
    print(f"{wind_angle_deg:8.1f} {boom_angle_deg:9.1f} {boom_rate_deg_s:15.1f}")


In [ ]:
gybe_boat = Sailboat(
    sail=sail,
    boom=Boom(
        boom_length=SAIL_FOOT_M,
        min_sheet_len=0.0,
        max_sheet_len=2 * SAIL_FOOT_M * math.sin(math.pi / 4),
    ),
)
gybe_boat.true_wind_speed = 8.0
gybe_boat.boat_speed = 4.0
gybe_boat.boat_heading_rad = 0.0
gybe_boat.sheet_len = gybe_boat.boom.max_sheet_len

times_s = []
wind_deg_history = []
boom_deg_history = []
boom_rate_deg_s_history = []
torque_nm_history = []

dt_s = 0.02
wind_schedule_deg = [160.0] * 100 + [170.0] * 20 + [175.0] * 20 + [179.0] * 20 + [180.0] * 20 + [181.0] * 20 + [185.0] * 20 + [190.0] * 20 + [200.0] * 40

for wind_angle_deg in wind_schedule_deg:
    gybe_boat.true_wind_angle_rad = math.radians(wind_angle_deg)
    gybe_boat.step(dt_s)
    times_s.append(gybe_boat.time_s)
    wind_deg_history.append(wind_angle_deg)
    boom_deg_history.append(math.degrees(gybe_boat.boom_angle_rad()))
    boom_rate_deg_s_history.append(math.degrees(gybe_boat.boom_angular_velocity_rad_s))
    torque_nm_history.append(gybe_boat.net_boom_torque_nm())

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(times_s, boom_deg_history, color="tab:blue", linewidth=2)
axes[0].set_ylabel("Boom angle (deg)")
axes[0].set_title("Gybe Dynamics")
axes[0].grid(True, alpha=0.3)

axes[1].plot(times_s, boom_rate_deg_s_history, color="tab:orange", linewidth=2)
axes[1].axhline(0.0, color="0.7", linewidth=1)
axes[1].set_ylabel("Boom rate (deg/s)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(times_s, torque_nm_history, color="tab:green", linewidth=2, label="Net boom torque")
axes[2].axhline(0.0, color="0.7", linewidth=1)
axes[2].set_ylabel("Torque (N m)")
axes[2].set_xlabel("Time (s)")
axes[2].grid(True, alpha=0.3)

for ax in axes:
    last_wind_deg = wind_deg_history[0]
    for time_s, wind_angle_deg in zip(times_s, wind_deg_history):
        if wind_angle_deg != last_wind_deg:
            ax.axvline(time_s, color="0.85", linestyle="--", linewidth=1)
            last_wind_deg = wind_angle_deg

plt.tight_layout()
